# Plot component traces by stimulus

Loads the CNMF results saved from `explore_cnmf_components.ipynb`, lets you pick a subset of components, and plots their traces with stimulus periods marked.

Prerequisite: `cnmf_results.hdf5` exists in `data/processed/2026_05_19_animal_1/` (saved via `cnm.save(...)` at the end of `explore_cnmf_components.ipynb`).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from caiman.source_extraction.cnmf.cnmf import load_CNMF

## Load results

In [ ]:
output_dir = Path('../data/processed/2026_05_19_animal_1')
results_path = output_dir / 'cnmf_results.hdf5'

cnm = load_CNMF(str(results_path))
estimates = cnm.estimates
estimates.A.shape  # (n_pixels, n_components)

In [ ]:
# F_dff isn't saved by cnm.save() by default -- recompute if it's missing.
if estimates.F_dff is None:
    estimates.detrend_df_f()
estimates.F_dff.shape

## Define stimulus epochs

TODO: fill in with your actual stimulus timing (in frame numbers, relative to
the stitched movie -- see `stitch_order.txt` in the same output folder for
the run order used when concatenating). Each entry is one stimulus
presentation: (label, start_frame, end_frame).

In [ ]:
stimulus_epochs = [
    ('stim_A', 100, 150),
    ('stim_B', 300, 350),
    ('stim_A', 500, 550),
]

stimulus_colors = {
    'stim_A': 'tab:orange',
    'stim_B': 'tab:green',
}

## Select components to plot

TODO: replace with the components you actually care about -- e.g. a manually
curated list, or `estimates.idx_components` if you've already run
`evaluate_components`.

In [ ]:
component_subset = [0, 1, 2]

## Plot traces with stimulus periods marked

In [ ]:
def plot_traces_by_stimulus(estimates, component_subset, stimulus_epochs, stimulus_colors, trace='F_dff'):
    traces = getattr(estimates, trace)
    fig, axes = plt.subplots(len(component_subset), 1, figsize=(12, 2.5 * len(component_subset)), sharex=True)
    axes = np.atleast_1d(axes)

    labeled = set()
    for ax, idx in zip(axes, component_subset):
        ax.plot(traces[idx], color='black', linewidth=1)
        for label, start, end in stimulus_epochs:
            ax.axvspan(start, end, color=stimulus_colors.get(label, 'gray'), alpha=0.3,
                       label=label if label not in labeled else None)
            labeled.add(label)
        ax.set_ylabel(f'component {idx}')

    axes[0].legend(loc='upper right')
    axes[-1].set_xlabel('Frame')
    fig.suptitle(f'{trace} by stimulus')
    fig.tight_layout()
    return fig, axes


plot_traces_by_stimulus(estimates, component_subset, stimulus_epochs, stimulus_colors)